<a href="https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

> Add blockquote



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Checking two signals before building the rule

Before writing any rule, I'm testing whether the signals it would lean on actually behave the way intuition suggests — a lesson from my own Week 1 discovery, where "older pages decline more" turned out to be backwards.

Signal 1: Age / Staleness. This connects directly to FlyRank's real refresh-flag logic, which typically assumes older content is more likely to need updating. I group all 30,000 pages into their age tiers, count how many pages are in each tier (n), and calculate what percentage of pages in that tier are declining. This recreates the age-gradient table from my ML-02 analysis, this time as a formal, checked signal for this week's rule.

Signal 2: Impressions Volume. This connects to "quick-win" logic — the idea that a page already carrying meaningful traffic is a bigger opportunity to fix than a barely-visited one. I split all pages into four equal-sized traffic tiers (low, mid-low, mid-high, high) using quantile cuts, then check the decline rate within each tier. This tests whether a page's current size predicts its risk of declining, or whether it's better used purely for prioritizing which fixes matter most.

Both checks print n for each bucket, so the pattern (or lack of one) isn't based on a handful of pages — it's grounded in real counts across the full 30,000-page dataset.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/sarahibdah/Flyrank-ML-Internship-Sarah/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Signal 1: staleness/age — ties to refresh flags from the session
bucket1 = df.groupby("age_tier_order").agg(
    age_tier=("age_tier", "first"),
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda s: s.eq("down").mean())
)
print("Signal 1: Age / Staleness")
print(bucket1.to_string())

# Signal 2: current impressions volume — ties to "quick-win" logic
df["impressions_bucket"] = pd.qcut(df["impressions_90d"], q=4, labels=["low", "mid-low", "mid-high", "high"])
bucket2 = df.groupby("impressions_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda s: s.eq("down").mean())
)
print("\nSignal 2: Impressions Volume")
print(bucket2.to_string())

Signal 1: Age / Staleness
               age_tier      n  decline_rate
age_tier_order                              
3                 31-90    492      0.668699
4                91-180  11780      0.625552
5               181-365  11368      0.514866
6                  365+   6360      0.426258

Signal 2: Impressions Volume
                       n  decline_rate
impressions_bucket                    
low                 7503      0.376116
mid-low             7499      0.604614
mid-high            7498      0.625634
high                7500      0.562000


Signal 1 verdict: OPPOSITE. Decline rate falls steadily with age (66.9% → 62.6% → 51.5% → 42.6% across n=492/11,780/11,368/6,360). This directly contradicts the "older content needs refreshing" assumption behind typical staleness flags — it's a genuine save, since a rule flagging old pages as high-risk would target the wrong group entirely.

## Signal 2 verdict: MIXED. Decline rate isn't monotonic across traffic tiers (37.6% → 60.5% → 62.6% → 56.2%, n≈7,500 per tier). Low-traffic pages show notably lower decline rates, but there's no clean "more traffic = more/less risk" trend among the rest. Volume isn't a reliable risk signal by itself, but remains useful for weighting impact once risk is otherwise established.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os

# The rule: age drives risk (Signal 1), volume drives impact (Signal 2)
risky_age = df["age_tier_order"].isin([3, 4])  # 31-90 and 91-180 day tiers
is_declining = df["trend_direction"] == "down"

def assign_reason(row):
    if row["risky_age"] and row["is_declining"]:
        return "young_declining_high_stakes"
    elif row["is_declining"]:
        return "older_declining"
    else:
        return "stable"

def assign_action(reason):
    if reason == "young_declining_high_stakes":
        return "refresh_now"
    elif reason == "older_declining":
        return "monitor"
    else:
        return "no_action"

df["risky_age"] = risky_age
df["is_declining"] = is_declining
df["reason_code"] = df.apply(assign_reason, axis=1)
df["action"] = df["reason_code"].apply(assign_action)

# Score: urgency comes first (from the rule's priority), impressions break ties within it
urgency_rank = {"young_declining_high_stakes": 2, "older_declining": 1, "stable": 0}
df["urgency"] = df["reason_code"].map(urgency_rank)
df["score"] = df["urgency"] * 1_000_000 + df["impressions_90d"].where(df["reason_code"] != "stable", 0)

queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "age_tier", "impressions_90d", "reason_code", "action", "score"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue written: {len(queue):,} pages")
print(f"\nAction breakdown:")
print(queue["action"].value_counts())
print(f"\nTop 20:")
print(queue.head(20).to_string())

Queue written: 30,000 pages

Action breakdown:
action
no_action      13738
monitor         8564
refresh_now     7698
Name: count, dtype: int64

Top 20:
                 content_id          client_id age_tier  impressions_90d                  reason_code       action    score
26531  content_cb112fce36be  client_19581e27de   91-180           309910  young_declining_high_stakes  refresh_now  2309910
27478  content_008fb02c46cb  client_349c41201b   91-180           236803  young_declining_high_stakes  refresh_now  2236803
23767  content_813e88069237  client_6208ef0f77   91-180           233561  young_declining_high_stakes  refresh_now  2233561
26304  content_ff94c9b6b411  client_349c41201b   91-180           228566  young_declining_high_stakes  refresh_now  2228566
10741  content_07e0b9af8b1a  client_b4944c6ff0   91-180           214816  young_declining_high_stakes  refresh_now  2214816
11655  content_cea79ef51519  client_f369cb89fc   91-180           208798  young_declining_high_stakes  r

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_cb112fce36be (91-180d, 309,910 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: high — largest traffic at stake, clean risky-age match. Wrong if: decline is seasonal and recovers without intervention.

2. content_008fb02c46cb (91-180d, 236,803 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same client appears 5x in this top 20 (#2, #4, #7, #9, #15), suggesting a client-wide pattern rather than page-specific decay. Wrong if: this client had a site-wide migration or tracking change, not independent page decline.

3. content_813e88069237 (91-180d, 233,561 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — high traffic, but same client also appears at #17, #20. Wrong if: a newer page from this client now covers the same topic and traffic simply shifted (cannibalization).

4. content_ff94c9b6b411 (91-180d, 228,566 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same recurring client as #2. Wrong if: a client-wide GSC data issue is driving the apparent decline, not real traffic loss.

5. content_07e0b9af8b1a (91-180d, 214,816 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: high — distinct client, clear signal. Wrong if: a competitor recently outranked this page — needs new content/backlinks, not just a refresh.

6. content_cea79ef51519 (91-180d, 208,798 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same client recurs at #19. Wrong if: an editor already has this page mid-update and the rule doesn't know.

7. content_bf7bff5d0756 (91-180d, 197,199 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same recurring client as #2/#4/#9/#15. Wrong if: this client's whole site dipped together from one systemic issue.

8. content_3d94572c3a35 (91-180d, 190,623 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: high — distinct client, strong signal. Wrong if: the drop is a short-term algorithm fluctuation that self-corrects.

9. content_8ba747cf969e (91-180d, 152,968 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same recurring client as #2/#4/#7/#15. Wrong if: this page was already refreshed recently and the drop reflects a slow adoption curve.

10. content_11fcfd65d94c (91-180d, 149,083 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same client as #11/#13/#14. Wrong if: the topic itself has declining search demand overall — no refresh fixes a shrinking market.

11. content_97a86caf3a3d (91-180d, 147,670 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same recurring client as #10/#13/#14. Wrong if: the client discontinued the product/service this page covers.

12. content_453722754fea (91-180d, 140,079 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same client as #18/#20. Wrong if: the client is intentionally deprioritizing this topic (planned decline, not a problem).

13. content_c1fe78bc4e37 (91-180d, 134,055 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same recurring client as #10/#11/#14. Wrong if: seasonal — worth checking last year's same-period performance.

14. content_e9c6e67086f6 (91-180d, 126,441 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: low — 4th appearance of this client (#10/#11/#13/#14), strongly suggests a client-level cause, not independent page decay. Wrong if: one root cause affects all four pages and refreshing each individually is the wrong fix.

15. content_53f466b9f954 (91-180d, 125,859 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: low — 5th appearance of the #2/#4/#7/#9 client. Wrong if: this client has a systemic data or site issue driving all five flagged pages.

16. content_370de6e8e035 (91-180d, 114,389 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: high — distinct client. Wrong if: the page targets a highly competitive keyword where refresh effort has low ROI.

17. content_50426bec207f (91-180d, 114,048 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same client as #3/#20. Wrong if: a technical issue (broken links, slow load) is suppressing rankings, not content staleness.

18. content_12a1ed6d5a62 (91-180d, 113,535 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same client as #12/#20. Wrong if: the page lost a featured snippet to a competitor — a different fix than a refresh.

19. content_39881853ef0c (91-180d, 112,434 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: medium — same client as #6. Wrong if: same systemic pattern as #6 — worth investigating client-level, not page-level.

20. content_15bbc0978284 (91-180d, 109,577 impr) — Action: refresh_now. Reason: young_declining_high_stakes. Confidence: low — 3rd appearance of the #3/#17 client. Wrong if: one shared cause (e.g. site migration) explains all three flagged pages from this client.






In [ ]:
print("Top 20 pages under review:")
print(queue.head(20)[["content_id", "client_id", "age_tier", "impressions_90d", "reason_code", "action"]].to_string())

Top 20 pages under review:
                 content_id          client_id age_tier  impressions_90d                  reason_code       action
26531  content_cb112fce36be  client_19581e27de   91-180           309910  young_declining_high_stakes  refresh_now
27478  content_008fb02c46cb  client_349c41201b   91-180           236803  young_declining_high_stakes  refresh_now
23767  content_813e88069237  client_6208ef0f77   91-180           233561  young_declining_high_stakes  refresh_now
26304  content_ff94c9b6b411  client_349c41201b   91-180           228566  young_declining_high_stakes  refresh_now
10741  content_07e0b9af8b1a  client_b4944c6ff0   91-180           214816  young_declining_high_stakes  refresh_now
11655  content_cea79ef51519  client_f369cb89fc   91-180           208798  young_declining_high_stakes  refresh_now
16950  content_bf7bff5d0756  client_349c41201b   91-180           197199  young_declining_high_stakes  refresh_now
7133   content_3d94572c3a35  client_19581e27de   91-1

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick pattern — client clustering. Verified counts (computed below): client_19581e27de appears 7x, client_349c41201b appears 5x, client_6208ef0f77 and client_f369cb89fc each appear 3x. That means 18 of my 20 top picks come from just 4 clients. When one client dominates the top of the queue this heavily, it's a signal worth questioning: is this genuinely 7+ independently declining pages, or one client-wide event (a site migration, a GSC tracking gap, a redesign) that dropped many pages at once? My rule treats every page independently and can't tell the difference — a real editor would investigate the client level before treating these as 18 separate problems

In [ ]:
client_counts = queue.head(20)["client_id"].value_counts()
print("Client frequency in top 20:")
print(client_counts)

Client frequency in top 20:
client_id
client_19581e27de    7
client_349c41201b    5
client_6208ef0f77    3
client_f369cb89fc    3
client_b4944c6ff0    1
client_7f2253d7e2    1
Name: count, dtype: int64


In [ ]:
rule_inputs = ["age_tier_order", "trend_direction", "impressions_90d"]
print(f"Columns used by the rule's logic: {rule_inputs}")
print(f"\nVerification:")
print(f"- age_tier_order: fixed property of the page, known at any point — not future, not a flag")
print(f"- trend_direction: built from impressions_prev_30d vs impressions_last_30d, both already-elapsed windows")
print(f"- impressions_90d: trailing historical metric, not a future-window value")
print(f"\nNone of these are pre-existing product/editor decision flags (e.g. no *_flag, *_priority, or")
print(f"*_action columns from the raw data were used) — the rule derives its own conclusion from raw metrics.")

Columns used by the rule's logic: ['age_tier_order', 'trend_direction', 'impressions_90d']

Verification:
- age_tier_order: fixed property of the page, known at any point — not future, not a flag
- trend_direction: built from impressions_prev_30d vs impressions_last_30d, both already-elapsed windows
- impressions_90d: trailing historical metric, not a future-window value

None of these are pre-existing product/editor decision flags (e.g. no *_flag, *_priority, or
*_action columns from the raw data were used) — the rule derives its own conclusion from raw metrics.


Leakage check: my rule's three inputs — age_tier_order, trend_direction, impressions_90d — are all pre-existing, already-elapsed metrics knowable at any point in time, not future-window data. None come from a product/editor decision column (like a pre-existing flag or prior refresh action) that would let the rule simply copy an existing human judgment rather than derive its own. Verified below by listing exactly which columns the rule touches.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.